<a href="https://colab.research.google.com/github/devendrasinghr284/NLP_Customer_Dispute_Prediction/blob/main/RAG_Development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain sentence-transformers faiss-cpu pypdf transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 378.1/378.1 kB 14.8 MB/s eta 0:00:00


In [4]:
#Step 3: Upload Your Research Papers

from google.colab import files
uploaded = files.upload()



Saving 1706.03762v7.pdf to 1706.03762v7.pdf
Saving 2005.11401v4.pdf to 2005.11401v4.pdf
Saving 2005.14165v4.pdf to 2005.14165v4.pdf


In [5]:
#Step 4: Extract Text from PDFs

from pypdf import PdfReader
import os

documents = []

for file_name in uploaded.keys():
  reader = PdfReader(file_name)

full_text = ""

for page in reader.pages:
  text = page.extract_text()
  if text:
    full_text += text + "\\n"

    documents.append({ "source": file_name, "text": full_text })

    print(f"Loaded {len(documents)} documents")

Loaded 1 documents
Loaded 2 documents
Loaded 3 documents
Loaded 4 documents
Loaded 5 documents
Loaded 6 documents
Loaded 7 documents
Loaded 8 documents
Loaded 9 documents
Loaded 10 documents
Loaded 11 documents
Loaded 12 documents
Loaded 13 documents
Loaded 14 documents
Loaded 15 documents
Loaded 16 documents
Loaded 17 documents
Loaded 18 documents
Loaded 19 documents
Loaded 20 documents
Loaded 21 documents
Loaded 22 documents
Loaded 23 documents
Loaded 24 documents
Loaded 25 documents
Loaded 26 documents
Loaded 27 documents
Loaded 28 documents
Loaded 29 documents
Loaded 30 documents
Loaded 31 documents
Loaded 32 documents
Loaded 33 documents
Loaded 34 documents
Loaded 35 documents
Loaded 36 documents
Loaded 37 documents
Loaded 38 documents
Loaded 39 documents
Loaded 40 documents
Loaded 41 documents
Loaded 42 documents
Loaded 43 documents
Loaded 44 documents
Loaded 45 documents
Loaded 46 documents
Loaded 47 documents
Loaded 48 documents
Loaded 49 documents
Loaded 50 documents
Loaded 51

In [8]:
!pip install -q langchain langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [15]:
#Step 5: Split the Documents into Chunks


splitter = RecursiveCharacterTextSplitter( chunk_size=5000, chunk_overlap=800 )

chunks = []

for doc in documents:
  split_texts = splitter.split_text(doc["text"])

  for i, chunk in enumerate(split_texts):
    chunks.append({
        "source": doc["source"],
        "chunk_id": i,
        "text": chunk
        })

print(f"Created {len(chunks)} chunks")

Created 2266 chunks


In [16]:
from sentence_transformers import SentenceTransformer


embedding_model = SentenceTransformer( "sentence-transformers/all-MiniLM-L6-v2" )

texts = [c["text"] for c in chunks]

embeddings = embedding_model.encode( texts, show_progress_bar=True )

print(embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/71 [00:00<?, ?it/s]

(2266, 384)


In [17]:
#Step 7: Build the Vector Database (FAISS)


import faiss
import numpy as np

In [18]:
dimension = embeddings.shape[1]

In [19]:
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))
print(f"FAISS index contains {index.ntotal} vectors")


FAISS index contains 2266 vectors


In [21]:
import numpy as np

def retrieve(query, top_k=5):
    # Encode the query
    query_embedding = embedding_model.encode([query])

    # Search in the FAISS index
    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        top_k
    )

    # Collect retrieved chunks
    results = []
    for idx in indices[0]:
        results.append(chunks[idx])

    return results


# Test retrieval
results = retrieve("What is multi-head attention?", top_k=3)

# Print results
for i, res in enumerate(results, 1):
    print(f"Result {i}:\\n{res}\\n")





Result 1:\n{'source': '2005.14165v4.pdf', 'chunk_id': 33, 'text': 'multi-task ﬁne-tuning rather than for in-context learning without weight updates.\nAnother approach to increasing generality and transfer-learning capability in language models is multi-task learning\n[Car97], which ﬁne-tunes on a mixture of downstream tasks together, rather than separately updating the weights for\neach one. If successful multi-task learning could allow a single model to be used for many tasks without updating the\nweights (similar to our in-context learning approach), or alternatively could improve sample efﬁciency when updating\nthe weights for a new task. Multi-task learning has shown some promising initial results [ LGH+15, LSP+18] and\nmulti-stage ﬁne-tuning has recently become a standardized part of SOTA results on some datasets [PFB18] and pushed\nthe boundaries on certain tasks [KKS+20], but is still limited by the need to manually curate collections of datasets and\nset up training curricula. 

In [24]:
#Step 9: Load an LLM for Answer Generation


from transformers import pipeline

generator = pipeline( "text-generation",
                     model="google/flan-t5-base", max_new_tokens=256 )





model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', '

In [29]:
#Step 10: Build the RAG Prompt

def build_prompt(question, retrieved_docs):
    context = "\\n\\n".join([ f"Source: {d['source']}\\n{d['text']}" for d in retrieved_docs ])
    prompt = f"""
    You are a research assistant.
    Answer the question using ONLY the provided Context.

    Context: {Context}

    Question: {question}

    Answer:
    """

    return prompt




In [30]:
#Step 11: Generate the Final Answer

def answer_question(question, top_k=5):
    retrieved_docs = retrieve(question, top_k)
    prompt = build_prompt(question, retrieved_docs)
    response = generator(prompt)[0]["generated_text"]
    sources = list(set([ d["source"] for d in retrieved_docs ]))

    return { "question": question, "answer": response, "sources": sources, "retrieved_docs": retrieved_docs }


In [32]:
def answer_question(question, top_k=3):
    # Retrieve relevant documents
    retrieved_docs = retrieve(question, top_k=top_k)

    # Build context from retrieved documents
    context = "\n\n".join([doc["text"] for doc in retrieved_docs])

    # Create prompt
    prompt = f"""
Use the following context to answer the question.

Context:
{context}

Question: {question}

Answer:
"""

    # Generate answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = model.generate(**inputs, max_new_tokens=120)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer,
        "sources": list(set([doc["source"] for doc in retrieved_docs])),
        "retrieved_docs": retrieved_docs
    }

In [34]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load FLAN-T5 model
model_name = "google/flan-t5-base"


tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Tokenizer and model loaded successfully!")


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Tokenizer and model loaded successfully!


In [35]:
questions = [
    "What are the main components of a RAG model, and how do they interact?",
    "What are the two sub-layers in each encoder layer of the Transformer model?",
    "Explain how positional encoding is implemented in Transformers and why it is necessary.",
    "Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?",
    "What is few-shot learning, and how does GPT-3 implement it during inference?"
]

for q in questions:
    result = answer_question(q)

    print("=" * 80)
    print("Q:", q)
    print("\nA:", result["answer"])
    print("\nSources:", result["sources"])
    print()

Q: What are the main components of a RAG model, and how do they interact?

A: gay marriage.

Sources: ['2005.14165v4.pdf']

Q: What are the two sub-layers in each encoder layer of the Transformer model?

A: training, the model is designed to be large to absorb information during pre-training, but are then fine-tuned on very narrow task distributions

Sources: ['2005.14165v4.pdf']

Q: Explain how positional encoding is implemented in Transformers and why it is necessary.

A: [ADG+16] Marcin Andrychowicz, Misha Denil, Sergio Gomez, Matthew W Hoffman, David Pfau, Tom Schaul, Brendan Shillingford, and Nando De Freitas. Learning to learn by gradient descent by gradient descent. In Proceedings of the 2013 conference on empirical methods in natural language processing, pages 1533–1544, 2013. [BDD+09] Luisa Bentivogli, Ido Dagan, Hoa Trang Dang, Danilo Giampic

Sources: ['2005.14165v4.pdf']

Q: Describe the concept of multi-head attention in the Transformer architecture. Why is it beneficial?


In [36]:
print(tokenizer)
print(model)

T5Tokenizer(name_or_path='google/flan-t5-base', vocab_size=32100, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '<pad>'}, added_tokens_decoder={
	0: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32000: AddedToken("<extra_id_99>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32001: AddedToken("<extra_id_98>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32002: AddedToken("<extra_id_97>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	32003: AddedToken("<extra_id_96>", rstrip=False, lstrip=False, single_word=False, normalized=False, 